In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Helper to convert min/max timestamps to Asia/Shanghai timezone strings
def tz_min_max(ts_series):
    if ts_series.empty:
        return (None, None)
    # convert seconds -> UTC, then convert to Asia/Shanghai
    dt = pd.to_datetime(ts_series.dropna().astype('int64'), unit='s', utc=True)
    if dt.empty:
        return (None, None)
    min_ts = dt.min().tz_convert(tz)
    max_ts = dt.max().tz_convert(tz)
    fmt = "%Y-%m-%d %H:%M:%S %Z"
    return (min_ts.strftime(fmt), max_ts.strftime(fmt))

# Helper to prepare unique cmdb_id display (all if <=20 else first 20 + count)
def summarize_cmdb_ids(df):
    if 'cmdb_id' not in df.columns:
        return {'unique_count': 0, 'sample_list': []}
    unique = list(pd.unique(df['cmdb_id'].dropna()))
    uc = len(unique)
    sample = unique if uc <= 20 else unique[:20]
    return {'unique_count': uc, 'sample_list': sample}

# Helper to summarize name-like column with 50-limit rule
def summarize_name_column(df, colname):
    if colname not in df.columns:
        return {'unique_count': 0, 'sample_list': []}
    unique = list(pd.unique(df[colname].dropna()))
    uc = len(unique)
    sample = unique if uc <= 50 else unique[:50]
    return {'unique_count': uc, 'sample_list': sample}

# Summaries for each file
m_count = df_metric.shape[0]
m_min, m_max = tz_min_max(df_metric['timestamp']) if 'timestamp' in df_metric.columns else (None, None)
m_cmdb = summarize_cmdb_ids(df_metric)
m_kpi = summarize_name_column(df_metric, 'kpi_name')

t_count = df_trace.shape[0]
t_min, t_max = tz_min_max(df_trace['timestamp']) if 'timestamp' in df_trace.columns else (None, None)
t_cmdb = summarize_cmdb_ids(df_trace)
t_trace_names = summarize_name_column(df_trace, 'trace_name')

l_count = df_log.shape[0]
l_min, l_max = tz_min_max(df_log['timestamp']) if 'timestamp' in df_log.columns else (None, None)
l_cmdb = summarize_cmdb_ids(df_log)
l_log_names = summarize_name_column(df_log, 'log_name')

e_count = df_err.shape[0]
e_min, e_max = tz_min_max(df_err['timestamp']) if 'timestamp' in df_err.columns else (None, None)
e_cmdb = summarize_cmdb_ids(df_err)
# error_logs has no KPI column; leave empty
e_names = {'unique_count': 0, 'sample_list': []}

# Build a compact pandas DataFrame summary
summary_df = pd.DataFrame([
    {
        'file': 'metric.csv',
        'row_count': m_count,
        'min_timestamp_ASIA/Shanghai': m_min,
        'max_timestamp_ASIA/Shanghai': m_max,
        'unique_cmdb_count': m_cmdb['unique_count'],
        'unique_cmdb_sample': m_cmdb['sample_list'],
        'unique_name_count': m_kpi['unique_count'],
        'unique_name_sample': m_kpi['sample_list'] if m_kpi['sample_list'] else []
    },
    {
        'file': 'trace.csv',
        'row_count': t_count,
        'min_timestamp_ASIA/Shanghai': t_min,
        'max_timestamp_ASIA/Shanghai': t_max,
        'unique_cmdb_count': t_cmdb['unique_count'],
        'unique_cmdb_sample': t_cmdb['sample_list'],
        'unique_name_count': t_trace_names['unique_count'],
        'unique_name_sample': t_trace_names['sample_list'] if t_trace_names['sample_list'] else []
    },
    {
        'file': 'log.csv',
        'row_count': l_count,
        'min_timestamp_ASIA/Shanghai': l_min,
        'max_timestamp_ASIA/Shanghai': l_max,
        'unique_cmdb_count': l_cmdb['unique_count'],
        'unique_cmdb_sample': l_cmdb['sample_list'],
        'unique_name_count': l_log_names['unique_count'],
        'unique_name_sample': l_log_names['sample_list'] if l_log_names['sample_list'] else []
    },
    {
        'file': 'error_logs.csv',
        'row_count': e_count,
        'min_timestamp_ASIA/Shanghai': e_min,
        'max_timestamp_ASIA/Shanghai': e_max,
        'unique_cmdb_count': e_cmdb['unique_count'],
        'unique_cmdb_sample': e_cmdb['sample_list'],
        'unique_name_count': e_names['unique_count'],
        'unique_name_sample': e_names['sample_list']
    }
])

# Prepare up to 5 samples from error_logs with message truncated to 200 chars
if {'timestamp','cmdb_id','message'}.issubset(df_err.columns):
    err_samples = df_err[['timestamp','cmdb_id','message']].copy()
    # Convert timestamp for display in Asia/Shanghai too (compact)
    err_samples['timestamp_ASIA/Shanghai'] = pd.to_datetime(err_samples['timestamp'].astype('int64'), unit='s', utc=True).dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")
    err_samples['message_trunc'] = err_samples['message'].astype(str).str.slice(0,200)
    err_samples = err_samples[['timestamp_ASIA/Shanghai','cmdb_id','message_trunc']].head(5)
else:
    # If columns missing, create empty compact DF
    err_samples = pd.DataFrame(columns=['timestamp_ASIA/Shanghai','cmdb_id','message_trunc'])

# Keep display compact: show summary_df and err_samples (up to 5 rows)
summary_df, err_samples

```
Out[1]:
```
```python
# Summarize the previously computed summary_df and err_samples in plain English.
# Reuse existing variables from prior execution: summary_df, err_samples

row = lambda fname: summary_df[summary_df['file']==fname].iloc[0]

m = row('metric.csv')
t = row('trace.csv')
l = row('log.csv')
e = row('error_logs.csv')

# Helper to format sample lists compactly
def fmt_list(lst, max_items=20):
    if not lst:
        return "[]"
    shown = lst if len(lst) <= max_items else lst[:max_items]
    s = ", ".join(map(str, shown))
    if len(lst) > max_items:
        s += f", ... (total {len(lst)} items)"
    return f"[{s}]"

summary = (
    "Summary of loaded telemetry files (all times in Asia/Shanghai):\n\n"
    "1) metric.csv\n"
    f"   - Total rows: {int(m['row_count'])}\n"
    f"   - Time range: {m['min_timestamp_ASIA/Shanghai']} to {m['max_timestamp_ASIA/Shanghai']}\n"
    f"   - Unique cmdb_id count: {int(m['unique_cmdb_count'])}\n"
    f"   - cmdb_id sample: {fmt_list(m['unique_cmdb_sample'], max_items=20)}\n"
    f"   - Unique KPI names: {int(m['unique_name_count'])} (showing first 50 below)\n"
    f"   - KPI name sample (first 50): {fmt_list(m['unique_name_sample'], max_items=50)}\n\n"
    "2) trace.csv\n"
    f"   - Total rows: {int(t['row_count'])}\n"
    f"   - Time range: {t['min_timestamp_ASIA/Shanghai']} to {t['max_timestamp_ASIA/Shanghai']}\n"
    f"   - Unique cmdb_id count: {int(t['unique_cmdb_count'])}\n"
    f"   - cmdb_id sample: {fmt_list(t['unique_cmdb_sample'], max_items=20)}\n"
    f"   - Unique trace names: {int(t['unique_name_count'])} (showing first 50 below)\n"
    f"   - Trace name sample (first 50): {fmt_list(t['unique_name_sample'], max_items=50)}\n\n"
    "3) log.csv\n"
    f"   - Total rows: {int(l['row_count'])}\n"
    f"   - Time range: {l['min_timestamp_ASIA/Shanghai']} to {l['max_timestamp_ASIA/Shanghai']}\n"
    f"   - Unique cmdb_id count: {int(l['unique_cmdb_count'])}\n"
    f"   - cmdb_id sample: {fmt_list(l['unique_cmdb_sample'], max_items=20)}\n"
    f"   - Unique log names: {int(l['unique_name_count'])}\n"
    f"   - Log name sample: {fmt_list(l['unique_name_sample'], max_items=50)}\n\n"
    "4) error_logs.csv\n"
    f"   - Total rows: {int(e['row_count'])}\n"
    f"   - Time range: {e['min_timestamp_ASIA/Shanghai']} to {e['max_timestamp_ASIA/Shanghai']}\n"
    f"   - Unique cmdb_id count: {int(e['unique_cmdb_count'])}\n"
    f"   - Sample rows (up to 5): {err_samples.shape[0]} shown below (timestamp, cmdb_id, message truncated to 200 chars)\n"
)

# Display the summary string (compact, plain English)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file  row_count min_timestamp_ASIA/Shanghai max_timestamp_ASIA/Shanghai  unique_cmdb_count                                 unique_cmdb_sample  unique_name_count                                 unique_name_sample
0      metric.csv      21838     2021-03-23 18:00:00 CST     2021-03-23 18:30:00 CST                 25  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...                273  [JVM-Memory_7778_JVM_Memory_HeapMemoryMax, JVM...
1       trace.csv        264     1970-01-20 01:01:00 CST     1970-01-20 01:01:00 CST                 12  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...                 72  [trace.from_root.duration_mean, trace.from_roo...
2         log.csv        480     2021-03-23 18:00:00 CST     2021-03-23 18:30:00 CST                  7  [Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat...                  3  [log.error_count, log.row_count, log.gc_pause_...
3  error_logs.csv          0                        None                        None                  0                                                 []                  0                                                 [], Empty DataFrame
Columns: [timestamp_ASIA/Shanghai, cmdb_id, message_trunc]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz, datetime

# Reuse df_metric and tz if available in kernel; otherwise reload
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 per (cmdb_id, kpi_name) across entire metric.csv
thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 2) Filter metric.csv to the incident window (inclusive) in Asia/Shanghai
start_local = tz.localize(datetime.datetime(2021, 3, 23, 18, 0, 0))
end_local = tz.localize(datetime.datetime(2021, 3, 23, 18, 30, 0))
# Convert to UTC epoch seconds for comparison (timestamps in metric are epoch seconds)
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge thresholds into windowed data so we can mark anomalies using precomputed P95
df_w = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies where value >= global_P95 (if global_P95 is NaN, treat as non-anomaly)
df_w['is_anomaly'] = False
mask_has_threshold = df_w['global_P95'].notna()
df_w.loc[mask_has_threshold, 'is_anomaly'] = df_w.loc[mask_has_threshold, 'value'] >= df_w.loc[mask_has_threshold, 'global_P95']

# 3) Aggregate per series within the window
# a) anomaly counts and max value within window
stats_window = (
    df_w
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomaly_count_within_window=('is_anomaly', 'sum'),
        max_value_within_window=('value', 'max')
    )
)

# b) earliest anomaly timestamp (if any)
anom_rows = df_w[df_w['is_anomaly']]
earliest = (
    anom_rows
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp': 'earliest_anom_ts'})
)

# Merge stats and earliest times with the full thresholds list to include series that had no points in window
merged = thresholds.merge(stats_window, on=['cmdb_id', 'kpi_name'], how='left')
merged = merged.merge(earliest, on=['cmdb_id', 'kpi_name'], how='left')

# Fill missing numeric aggregations
merged['anomaly_count_within_window'] = merged['anomaly_count_within_window'].fillna(0).astype(int)
# max_value_within_window may remain NaN if no points within window; keep NaN to indicate no data
# Convert earliest_anom_ts to Asia/Shanghai formatted string (or None)
merged['earliest_anomaly_timestamp_ASIA/Shanghai'] = None
mask = merged['earliest_anom_ts'].notna()
if mask.any():
    merged.loc[mask, 'earliest_anomaly_timestamp_ASIA/Shanghai'] = pd.to_datetime(
        merged.loc[mask, 'earliest_anom_ts'].astype(int), unit='s', utc=True
    ).dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")

# Prepare final report columns
final = merged[[
    'cmdb_id',
    'kpi_name',
    'global_P95',
    'anomaly_count_within_window',
    'earliest_anomaly_timestamp_ASIA/Shanghai',
    'max_value_within_window'
]].copy()

# 4) Sort and limit to top 20 rows
final_sorted = final.sort_values(by=['anomaly_count_within_window', 'global_P95'], ascending=[False, False])
result_top20 = final_sorted.head(20)

# Total distinct series processed
total_series_count = int(thresholds.shape[0])

# Display compact results: top 20 rows and total series count
result_top20, total_series_count

```
Out[2]:
```
```python
# Build a plain-English summary of the previously computed results.
# Reuse kernel variables: result_top20, total_series_count

top_df = result_top20.copy()
n_display = min(10, top_df.shape[0])

lines = []
lines.append(f"Total distinct (cmdb_id, kpi_name) series processed: {total_series_count}")
lines.append(f"Top {n_display} series by anomaly count within 2021-03-23 18:00–18:30 (Asia/Shanghai):")

for i, row in enumerate(top_df.head(n_display).itertuples(index=False), start=1):
    cmdb_id = row.cmdb_id
    kpi = (row.kpi_name[:120] + '...') if len(row.kpi_name) > 120 else row.kpi_name
    gp95 = f"{row.global_P95:.6g}" if pd.notna(row.global_P95) else "NaN"
    ac = int(row.anomaly_count_within_window)
    earliest = row.earliest_anomaly_timestamp_ASIA_Shanghai if 'earliest_anomaly_timestamp_ASIA/Shanghai' not in top_df.columns else row._asdict().get('earliest_anomaly_timestamp_ASIA/Shanghai', None)
    # handle potential column name differing in tuple keys
    if earliest is None and 'earliest_anomaly_timestamp_ASIA/Shanghai' in top_df.columns:
        earliest = row._asdict().get('earliest_anomaly_timestamp_ASIA/Shanghai')
    maxv = f"{row.max_value_within_window:.6g}" if pd.notna(row.max_value_within_window) else "No data in window"
    lines.append(f"{i}. {cmdb_id} | {kpi} | global_P95={gp95} | anomalies_in_window={ac} | earliest_anomaly={earliest} | max_in_window={maxv}")

lines.append("")
lines.append("Notes:")
lines.append("- The incident window is inclusive and spans 18:00 to 18:30 Asia/Shanghai (31 minutes).")
lines.append("- An anomaly is defined as a point within the window whose value >= the series' global P95 computed from the full metric.csv.")
lines.append("- Several series show anomaly_count=31, meaning every minute in the window exceeded the global P95 (earliest anomaly at 2021-03-23 18:00:00 CST).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                                           kpi_name    global_P95  anomaly_count_within_window earliest_anomaly_timestamp_ASIA/Shanghai  max_value_within_window
0             IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                           31                  2021-03-23 18:00:00 CST             2.040136e+09
18            IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                           31                  2021-03-23 18:00:00 CST             2.040136e+09
36            MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                           31                  2021-03-23 18:00:00 CST             2.040136e+09
140           MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                           31                  2021-03-23 18:00:00 CST             2.040136e+09
39            MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.907150e+07                           31                  2021-03-23 18:00:00 CST             5.907150e+07
3             IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.865392e+07                           31                  2021-03-23 18:00:00 CST             5.865392e+07
21            IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.854114e+07                           31                  2021-03-23 18:00:00 CST             5.854114e+07
996       Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  4.016560e+05                           31                  2021-03-23 18:00:00 CST             4.016560e+05
755       Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.978080e+05                           31                  2021-03-23 18:00:00 CST             3.978080e+05
874       Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.906190e+05                           31                  2021-03-23 18:00:00 CST             3.906190e+05
1117      Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  2.304870e+05                           31                  2021-03-23 18:00:00 CST             2.304870e+05
995       Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  6.210000e+02                           31                  2021-03-23 18:00:00 CST             6.210000e+02
754       Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.600000e+02                           31                  2021-03-23 18:00:00 CST             5.600000e+02
873       Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.150000e+02                           31                  2021-03-23 18:00:00 CST             5.150000e+02
769       Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                           31                  2021-03-23 18:00:00 CST             5.000000e+02
891       Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                           31                  2021-03-23 18:00:00 CST             5.000000e+02
1013      Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                           31                  2021-03-23 18:00:00 CST             5.000000e+02
1134      Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                           31                  2021-03-23 18:00:00 CST             5.000000e+02
1116      Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  1.390000e+02                           31                  2021-03-23 18:00:00 CST             1.390000e+02
608   ServiceTest1                                             app.rr  1.000000e+02                           31                  2021-03-23 18:00:00 CST             1.000000e+02, 1157)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz, datetime

# reuse df_log and tz if available, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

tz = pytz.timezone('Asia/Shanghai')

# Define inclusive incident window in Asia/Shanghai and convert to epoch seconds (UTC)
start_local = tz.localize(datetime.datetime(2021, 3, 23, 18, 0, 0))
end_local = tz.localize(datetime.datetime(2021, 3, 23, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter logs to the incident window (inclusive)
df_log_window = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# Ensure value numeric
df_log_window['value'] = pd.to_numeric(df_log_window['value'], errors='coerce')

# 1) Aggregates per cmdb_id and log_name within window
agg_log_full = (
    df_log_window
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        row_count=('value', 'size'),
        sum_value=('value', 'sum'),
        max_value=('value', 'max')
    )
)

# earliest timestamp where value > 0 (per group)
positive = df_log_window[df_log_window['value'] > 0].copy()
if not positive.empty:
    earliest_pos = (
        positive
        .groupby(['cmdb_id', 'log_name'], as_index=False)
        .agg(earliest_ts=('timestamp', 'min'))
    )
    # convert to Asia/Shanghai formatted string
    earliest_pos['earliest_timestamp_ASIA/Shanghai'] = pd.to_datetime(earliest_pos['earliest_ts'].astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")
    earliest_pos = earliest_pos.drop(columns=['earliest_ts'])
else:
    earliest_pos = pd.DataFrame(columns=['cmdb_id','log_name','earliest_timestamp_ASIA/Shanghai'])

# Merge earliest timestamps into full aggregate
agg_log_full = agg_log_full.merge(earliest_pos, on=['cmdb_id','log_name'], how='left')

# If no earliest timestamp, keep as None/NaN -> set to None for clarity
agg_log_full['earliest_timestamp_ASIA/Shanghai'] = agg_log_full['earliest_timestamp_ASIA/Shanghai'].where(agg_log_full['earliest_timestamp_ASIA/Shanghai'].notna(), None)

# Sort by sum_value desc and limit to top 20
agg_top20 = agg_log_full.sort_values(by='sum_value', ascending=False).head(20)

# 2) Sample up to 20 raw log rows in window for Tomcat01-04 and any existing log names
tomcats = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04']
# ensure we only include log_names that exist in the file (they do)
existing_log_names = df_log['log_name'].dropna().unique().tolist()

df_sample_raw = df_log_window[
    (df_log_window['cmdb_id'].isin(tomcats)) &
    (df_log_window['log_name'].isin(existing_log_names))
].copy()

# Convert timestamp to Asia/Shanghai formatted string for display
if not df_sample_raw.empty:
    df_sample_raw['timestamp_ASIA/Shanghai'] = pd.to_datetime(df_sample_raw['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")
else:
    df_sample_raw['timestamp_ASIA/Shanghai'] = []

# Select requested columns and limit to 20 rows
sample_raw = df_sample_raw[['timestamp_ASIA/Shanghai','cmdb_id','log_name','value']].head(20)

# Display compact outputs: aggregated top 20 and sampled raw rows
agg_top20, sample_raw

```
Out[3]:
```
```python
# Build a plain-English summary of the log.csv window analysis using existing kernel variables:
# agg_top20 (aggregated top rows) and sample_raw (sample raw rows).
# Reuse them from the kernel; produce a compact textual summary string as the result.

try:
    agg_top20
    sample_raw
except NameError:
    raise RuntimeError("Required variables agg_top20 and sample_raw not found in kernel.")

lines = []
lines.append("Log aggregates within 2021-03-23 18:00–18:30 (Asia/Shanghai):")
lines.append("")
lines.append("Top aggregated (cmdb_id, log_name) rows (sorted by sum_value desc, up to 20):")
for i, r in agg_top20.reset_index(drop=True).iterrows():
    idx = i + 1
    cmdb = r['cmdb_id']
    name = r['log_name']
    row_count = int(r['row_count']) if pd.notna(r['row_count']) else 0
    sumv = f"{r['sum_value']:.2f}" if pd.notna(r['sum_value']) else "0"
    maxv = f"{r['max_value']:.2f}" if pd.notna(r['max_value']) else "NaN"
    # safe access for earliest timestamp (column name contains '/')
    earliest_col = 'earliest_timestamp_ASIA/Shanghai'
    earliest = r.get(earliest_col, None)
    lines.append(f"{idx}. {cmdb} | {name} | row_count={row_count} | sum_value={sumv} | max_value={maxv} | earliest_positive_value={earliest}")

lines.append("")
lines.append(f"Sample raw log rows (filtered to Tomcat01-04, up to {min(20, len(sample_raw))} rows):")
# Show each sampled raw row: timestamp_ASIA/Shanghai, cmdb_id, log_name, value
for i, r in sample_raw.reset_index(drop=True).iterrows():
    ts = r.get('timestamp_ASIA/Shanghai', None)
    cmdb = r['cmdb_id']
    name = r['log_name']
    val = f"{r['value']:.2f}" if pd.notna(r['value']) else "NaN"
    lines.append(f"- {ts} | {cmdb} | {name} | value={val}")

lines.append("")
lines.append("Notes:")
lines.append("- Window is inclusive: 2021-03-23 18:00:00 to 2021-03-23 18:30:00 Asia/Shanghai (31 minutes).")
lines.append("- earliest_positive_value is the earliest timestamp in the window where value>0 for that (cmdb_id, log_name); 'None' means no positive values in the window.")
lines.append("- Several hosts (apache01/apache02 and Tomcat01-04) show row_count=31 (data present for every minute).")
lines.append("- For Tomcat servers, log.row_count sums are high (e.g., Tomcat03 sum=8201.00), while log.error_count is zero for these Tomcats in the window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  row_count  sum_value  max_value earliest_timestamp_ASIA/Shanghai
15  apache01     log.row_count         31   16029.00     676.00          2021-03-23 18:00:00 CST
17  apache02     log.row_count         31   15948.00     687.00          2021-03-23 18:00:00 CST
10  Tomcat03     log.row_count         31    8201.00     367.00          2021-03-23 18:00:00 CST
13  Tomcat04     log.row_count         31    8012.00     337.00          2021-03-23 18:00:00 CST
4   Tomcat01     log.row_count         31    7917.00     338.00          2021-03-23 18:00:00 CST
7   Tomcat02     log.row_count         31    7901.00     348.00          2021-03-23 18:00:00 CST
1    Mysql01     log.row_count         29     136.00      16.00          2021-03-23 18:00:00 CST
12  Tomcat04  log.gc_pause_max         14       0.95       0.09          2021-03-23 18:00:00 CST
6   Tomcat02  log.gc_pause_max         12       0.79       0.10          2021-03-23 18:01:00 CST
3   Tomcat01  log.gc_pause_max         12       0.76       0.08          2021-03-23 18:01:00 CST
9   Tomcat03  log.gc_pause_max         12       0.75       0.12          2021-03-23 18:01:00 CST
0    Mysql01   log.error_count         29       0.00       0.00                             None
2   Tomcat01   log.error_count         31       0.00       0.00                             None
5   Tomcat02   log.error_count         31       0.00       0.00                             None
8   Tomcat03   log.error_count         31       0.00       0.00                             None
11  Tomcat04   log.error_count         31       0.00       0.00                             None
14  apache01   log.error_count         31       0.00       0.00                             None
16  apache02   log.error_count         31       0.00       0.00                             None,     timestamp_ASIA/Shanghai   cmdb_id          log_name   value
2   2021-03-23 18:00:00 CST  Tomcat01   log.error_count    0.00
3   2021-03-23 18:00:00 CST  Tomcat01     log.row_count  294.00
4   2021-03-23 18:00:00 CST  Tomcat02   log.error_count    0.00
5   2021-03-23 18:00:00 CST  Tomcat02     log.row_count  308.00
6   2021-03-23 18:00:00 CST  Tomcat03   log.error_count    0.00
7   2021-03-23 18:00:00 CST  Tomcat03     log.row_count  327.00
8   2021-03-23 18:00:00 CST  Tomcat04   log.error_count    0.00
9   2021-03-23 18:00:00 CST  Tomcat04  log.gc_pause_max    0.08
10  2021-03-23 18:00:00 CST  Tomcat04     log.row_count  308.00
17  2021-03-23 18:01:00 CST  Tomcat01   log.error_count    0.00
18  2021-03-23 18:01:00 CST  Tomcat01  log.gc_pause_max    0.07
19  2021-03-23 18:01:00 CST  Tomcat01     log.row_count  311.00
20  2021-03-23 18:01:00 CST  Tomcat02   log.error_count    0.00
21  2021-03-23 18:01:00 CST  Tomcat02  log.gc_pause_max    0.02
22  2021-03-23 18:01:00 CST  Tomcat02     log.row_count  312.00
23  2021-03-23 18:01:00 CST  Tomcat03   log.error_count    0.00
24  2021-03-23 18:01:00 CST  Tomcat03  log.gc_pause_max    0.08
25  2021-03-23 18:01:00 CST  Tomcat03     log.row_count  312.00
26  2021-03-23 18:01:00 CST  Tomcat04   log.error_count    0.00
27  2021-03-23 18:01:00 CST  Tomcat04     log.row_count  329.00)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')
tomcats = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04']

# Reuse previously computed 'final' and 'df_w' if available
if 'final' not in globals():
    raise RuntimeError("Required aggregated anomaly table 'final' not found in kernel.")
if 'df_w' not in globals():
    raise RuntimeError("Required windowed merged table 'df_w' (with is_anomaly flags) not found in kernel.")

# 1) Filter to Tomcat series
tomcat_df = final[final['cmdb_id'].isin(tomcats)].copy()

# Ensure numeric columns
tomcat_df['global_P95'] = pd.to_numeric(tomcat_df['global_P95'], errors='coerce')
tomcat_df['max_value_within_window'] = pd.to_numeric(tomcat_df['max_value_within_window'], errors='coerce')

# 2) Compute breach_ratio
def compute_breach(row):
    gp = row['global_P95']
    mv = row['max_value_within_window']
    if pd.isna(gp) or gp == 0:
        return np.nan
    if pd.isna(mv):
        return np.nan
    return (mv - gp) / gp

tomcat_df['breach_ratio'] = tomcat_df.apply(compute_breach, axis=1)

# Truncate kpi_name to 120 chars for display
tomcat_df['kpi_name_trunc'] = tomcat_df['kpi_name'].astype(str).apply(lambda s: s if len(s) <= 120 else s[:120])

# Select and order columns as requested
tomcat_severity = tomcat_df[[
    'cmdb_id',
    'kpi_name_trunc',
    'global_P95',
    'max_value_within_window',
    'breach_ratio',
    'anomaly_count_within_window',
    'earliest_anomaly_timestamp_ASIA/Shanghai'
]].rename(columns={'kpi_name_trunc': 'kpi_name'})

# 3) Sort by breach_ratio desc, then anomaly_count desc, limit to top 20
tomcat_severity = tomcat_severity.sort_values(by=['breach_ratio', 'anomaly_count_within_window'], ascending=[False, False]).head(20)

# 4) Compute single earliest anomaly timestamp among all Tomcat series (use df_w is_anomaly rows)
df_w_tomcat_anoms = df_w[(df_w['cmdb_id'].isin(tomcats)) & (df_w['is_anomaly'] == True)].copy()
if not df_w_tomcat_anoms.empty:
    earliest_ts = int(df_w_tomcat_anoms['timestamp'].min())
    earliest_tomcat_start = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime("%Y-%m-%d %H:%M:%S %Z")
else:
    earliest_tomcat_start = None

# Keep outputs compact: display tomcat_severity (<=20 rows) and earliest_tomcat_start scalar
tomcat_severity, earliest_tomcat_start

```
Out[4]:
```
```python
# Summarize Tomcat severity ranking (plain English) using existing kernel variables:
# tomcat_severity (DataFrame) and earliest_tomcat_start (scalar).
# The summary string below is the only displayed output.

try:
    tomcat_severity
    earliest_tomcat_start
except NameError:
    raise RuntimeError("Required variables 'tomcat_severity' and 'earliest_tomcat_start' not found in kernel.")

lines = []
lines.append("Tomcat series severity ranking (based on metric-level anomalies within 2021-03-23 18:00–18:30 Asia/Shanghai):")
lines.append("")
rows_shown = tomcat_severity.shape[0]
lines.append(f"- Rows returned (up to 20): {rows_shown}")
lines.append(f"- Earliest Tomcat anomaly observed: {earliest_tomcat_start}")
lines.append("")
lines.append("Top series (sorted by breach_ratio desc, then anomaly_count desc):")
# Show each row in the provided tomcat_severity table (already sorted), up to 20 rows
for i, r in enumerate(tomcat_severity.itertuples(index=False), start=1):
    cmdb = r.cmdb_id
    kpi = (r.kpi_name[:120] + '...') if len(str(r.kpi_name)) > 120 else str(r.kpi_name)
    gp95 = f"{r.global_P95:.6g}" if pd.notna(r.global_P95) else "NaN"
    maxv = f"{r.max_value_within_window:.6g}" if pd.notna(r.max_value_within_window) else "No data"
    breach = f"{r.breach_ratio:.3f}" if pd.notna(r.breach_ratio) else "NaN"
    ac = int(r.anomaly_count_within_window) if pd.notna(r.anomaly_count_within_window) else 0
    earliest = r._asdict().get('earliest_anomaly_timestamp_ASIA/Shanghai', None)
    lines.append(f"{i}. {cmdb} | {kpi} | global_P95={gp95} | max_in_window={maxv} | breach_ratio={breach} | anomalies={ac} | earliest={earliest}")

lines.append("")
lines.append("Key observations:")
lines.append("- Most Tomcat anomalies are related to OS local disk metrics (DSKRead/DSKBps/DSKRTps) and thread counts.")
lines.append("- Highest breach_ratios reach ~3.0 (e.g., several DSK metrics on Tomcat04), indicating values ~3x above the series' global P95.")
lines.append("- Some Tomcat thread metrics show sustained anomalies (anomaly_count >1), with earliest anomaly at 2021-03-23 18:00:00 CST for some series.")
lines.append("")
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  global_P95  max_value_within_window  breach_ratio  anomaly_count_within_window earliest_anomaly_timestamp_ASIA/Shanghai
1067  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.250000                 1.000000      3.000000                            1                  2021-03-23 18:16:00 CST
1078  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    0.137500                 0.550000      3.000000                            1                  2021-03-23 18:06:00 CST
1079  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead    1.133333                 4.533333      3.000000                            1                  2021-03-23 18:06:00 CST
1069  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    1.662525                 6.000000      2.608968                            1                  2021-03-23 18:30:00 CST
825   Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    1.358325                 4.833300      2.558279                            1                  2021-03-23 18:16:00 CST
705   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.025000                 0.083333      2.333333                            1                  2021-03-23 18:22:00 CST
706   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    0.240000                 0.800000      2.333333                            1                  2021-03-23 18:22:00 CST
704   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.090000                 0.300000      2.333333                            1                  2021-03-23 18:24:00 CST
954   Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.600000                 2.000000      2.333333                            1                  2021-03-23 18:26:00 CST
1070  Tomcat04    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead   17.233325                50.933300      1.955512                            1                  2021-03-23 18:30:00 CST
826   Tomcat02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead   15.200000                42.800000      1.815789                            1                  2021-03-23 18:16:00 CST
946   Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.575000                 1.400000      1.434783                            1                  2021-03-23 18:30:00 CST
948   Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  107.480000               232.266667      1.161022                            1                  2021-03-23 18:01:00 CST
968   Tomcat03       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.500000                 1.000000      1.000000                            2                  2021-03-23 18:03:00 CST
837   Tomcat02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.550000                 1.000000      0.818182                            1                  2021-03-23 18:01:00 CST
1081  Tomcat04     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.550000                 1.000000      0.818182                            1                  2021-03-23 18:13:00 CST
708   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    0.600000                 1.000000      0.666667                            1                  2021-03-23 18:05:00 CST
950   Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    3.000000                 5.000000      0.666667                            1                  2021-03-23 18:03:00 CST
768   Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...    3.000000                 5.000000      0.666667                            9                  2021-03-23 18:00:00 CST
1012  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...    3.000000                 5.000000      0.666667                            4                  2021-03-23 18:00:00 CST, '2021-03-23 18:00:00 CST')```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz, datetime

tz = pytz.timezone('Asia/Shanghai')
components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04']

# Reuse df_log if available, else load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure numeric values
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Compute global P95 for log.row_count per cmdb_id using full log.csv
rowcount_full = df_log[df_log['log_name'] == 'log.row_count'].copy()
global_p95 = (
    rowcount_full
    .groupby('cmdb_id', as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# Ensure target components present
p95_target = pd.DataFrame({'cmdb_id': components}).merge(global_p95, on='cmdb_id', how='left')

# Define inclusive incident window and convert to epoch seconds (UTC)
start_local = tz.localize(datetime.datetime(2021, 3, 23, 18, 0, 0))
end_local = tz.localize(datetime.datetime(2021, 3, 23, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

# Filter to window
df_log_window = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# Focus on log.row_count within window for the components
window_rowcount = df_log_window[(df_log_window['log_name'] == 'log.row_count') & (df_log_window['cmdb_id'].isin(components))].copy()

# Aggregates: max and sum within window per component (may be missing if no rows)
agg_window = (
    window_rowcount
    .groupby('cmdb_id', as_index=False)
    .agg(
        max_log_row_count_within_window=('value', 'max'),
        sum_log_row_count_within_window=('value', 'sum')
    )
)

# Join p95 and ensure all components present
agg = p95_target.merge(agg_window, on='cmdb_id', how='left')

# Compute earliest timestamp in window where value >= global_P95 (work with epoch ints)
earliest_ts_map = {}
for cmdb in components:
    gp = agg.loc[agg['cmdb_id'] == cmdb, 'global_P95'].values[0]
    if pd.isna(gp):
        earliest_ts_map[cmdb] = None
        continue
    grp = window_rowcount[window_rowcount['cmdb_id'] == cmdb]
    if grp.empty:
        earliest_ts_map[cmdb] = None
        continue
    mask = grp['value'] >= gp
    if mask.any():
        earliest_ts_map[cmdb] = int(grp.loc[mask, 'timestamp'].min())
    else:
        earliest_ts_map[cmdb] = None

# Attach formatted earliest timestamps (Asia/Shanghai) to agg
def fmt_ts(ts_int):
    if ts_int is None or pd.isna(ts_int):
        return None
    return pd.to_datetime(int(ts_int), unit='s', utc=True).tz_convert(tz).strftime("%Y-%m-%d %H:%M:%S %Z")

agg['earliest_timestamp_in_window_where_log.row_count>=global_P95'] = agg['cmdb_id'].map(earliest_ts_map).map(fmt_ts)

# Reorder columns and keep only the six target rows
result_table = agg[[
    'cmdb_id',
    'global_P95',
    'earliest_timestamp_in_window_where_log.row_count>=global_P95',
    'max_log_row_count_within_window',
    'sum_log_row_count_within_window'
]].copy()

# Compute the single earliest timestamp among components (minimum of non-None earliest epoch ints)
valid_ts = [v for v in earliest_ts_map.values() if v is not None]
if valid_ts:
    earliest_crossing_any = pd.to_datetime(min(valid_ts), unit='s', utc=True).tz_convert(tz).strftime("%Y-%m-%d %H:%M:%S %Z")
else:
    earliest_crossing_any = None

# Store full intermediate aggregated table in kernel variable 'log_rowcount_agg_full'
log_rowcount_agg_full = agg.copy()

# Display compact results: the six-row table and the single earliest crossing timestamp
result_table, earliest_crossing_any

```
Out[5]:
```
```python
# Summarize the log.row_count P95 crossing results (plain English), reusing existing kernel variables:
# result_table and earliest_crossing_any produced earlier.

try:
    result_table
    earliest_crossing_any
except NameError:
    raise RuntimeError("Required variables 'result_table' and 'earliest_crossing_any' not found in kernel.")

lines = []
lines.append("Summary: log.row_count P95 vs incident window (2021-03-23 18:00–18:30 Asia/Shanghai)")
lines.append("")
lines.append("Per-component results (cmdb_id | global_P95 | earliest_timestamp_in_window_where log.row_count >= global_P95 | max_in_window | sum_in_window):")
for r in result_table.itertuples(index=False):
    cmdb = r.cmdb_id
    gp95 = f"{r.global_P95:.1f}" if pd.notna(r.global_P95) else "NaN"
    earliest = r[2]  # earliest_timestamp_in_window_where_log.row_count>=global_P95
    maxv = f"{r.max_log_row_count_within_window:.1f}" if pd.notna(r.max_log_row_count_within_window) else "No data"
    sumv = f"{r.sum_log_row_count_within_window:.1f}" if pd.notna(r.sum_log_row_count_within_window) else "No data"
    lines.append(f"- {cmdb} | global_P95={gp95} | earliest_crossing={earliest} | max={maxv} | sum={sumv}")

lines.append("")
lines.append(f"Overall earliest crossing among these components: {earliest_crossing_any}")
lines.append("")
lines.append("Notes:")
lines.append("- global_P95 computed from full log.csv per component for 'log.row_count'.")
lines.append("- Window is inclusive 2021-03-23 18:00:00 to 18:30:00 Asia/Shanghai.")
lines.append("- 'earliest_crossing' is None when the component did not reach its global P95 within the window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  global_P95 earliest_timestamp_in_window_where_log.row_count>=global_P95  max_log_row_count_within_window  sum_log_row_count_within_window
0  apache01       653.5                            2021-03-23 18:16:00 CST                                      676.0                          16029.0
1  apache02       647.0                            2021-03-23 18:16:00 CST                                      687.0                          15948.0
2  Tomcat01       332.5                            2021-03-23 18:16:00 CST                                      338.0                           7917.0
3  Tomcat02       320.5                            2021-03-23 18:16:00 CST                                      348.0                           7901.0
4  Tomcat03       343.5                            2021-03-23 18:03:00 CST                                      367.0                           8201.0
5  Tomcat04       325.5                            2021-03-23 18:01:00 CST                                      337.0                           8012.0, '2021-03-23 18:01:00 CST')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz, datetime

tz = pytz.timezone('Asia/Shanghai')
components = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04','apache01','apache02','IG01','IG02','MG01','MG02']

# Reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# 1) Compute global P95 per (cmdb_id, kpi_name) across full metric.csv (reuse 'thresholds' if exists)
if 'thresholds' in globals():
    metric_thresholds_global = thresholds.copy()
else:
    metric_thresholds_global = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_P95'})
    )

# 2) Filter metric.csv to the incident window (inclusive) in Asia/Shanghai -> convert to UTC epoch seconds
start_local = tz.localize(datetime.datetime(2021,3,23,18,0,0))
end_local = tz.localize(datetime.datetime(2021,3,23,18,30,0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds and mark anomalies where value >= global_P95 (use precomputed global thresholds)
metric_merged = metric_window.merge(metric_thresholds_global, on=['cmdb_id','kpi_name'], how='left')

# mark anomalies only when global_P95 exists and is not zero; if global_P95==0 set anomaly False to avoid division by zero
mask_has_thresh = metric_merged['global_P95'].notna()
metric_merged['is_anomaly'] = False
metric_merged.loc[mask_has_thresh, 'is_anomaly'] = metric_merged.loc[mask_has_thresh, 'value'] >= metric_merged.loc[mask_has_thresh, 'global_P95']

# 4) From anomaly points restrict to requested components
anomalies_all = metric_merged[metric_merged['is_anomaly']].copy()
anomalies_filtered = anomalies_all[anomalies_all['cmdb_id'].isin(components)].copy()

# compute percent_over safely (NaN if global_P95==0 or missing)
anomalies_filtered['percent_over'] = np.where(
    anomalies_filtered['global_P95'].replace(0, np.nan).notna(),
    (anomalies_filtered['value'] - anomalies_filtered['global_P95']) / anomalies_filtered['global_P95'],
    np.nan
)

# add formatted timestamp column
anomalies_filtered['timestamp_ASIA/Shanghai'] = pd.to_datetime(anomalies_filtered['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz).dt.strftime("%Y-%m-%d %H:%M:%S %Z")

# 5) earliest 5 anomaly points across these components sorted by timestamp ascending
anomalies_filtered_sorted = anomalies_filtered.sort_values(by='timestamp', ascending=True)
earliest5 = anomalies_filtered_sorted[['timestamp_ASIA/Shanghai','cmdb_id','kpi_name','value','global_P95','percent_over']].head(5).copy()

# single earliest anomaly timestamp among these components (formatted) or None
if not anomalies_filtered_sorted.empty:
    earliest_ts_int = int(anomalies_filtered_sorted['timestamp'].min())
    earliest_anomaly_timestamp_ASIA_Shanghai = pd.to_datetime(earliest_ts_int, unit='s', utc=True).tz_convert(tz).strftime("%Y-%m-%d %H:%M:%S %Z")
else:
    earliest_anomaly_timestamp_ASIA_Shanghai = None

# Store full intermediate tables in kernel variables (as requested)
metric_thresholds_global, metric_window, metric_merged, anomalies_all, anomalies_filtered, anomalies_filtered_sorted, earliest5, earliest_anomaly_timestamp_ASIA_Shanghai

```
Out[6]:
```
```python
# Summarize the earliest 5 anomaly points and the single earliest anomaly timestamp (plain English).
# Reuse existing kernel variables: earliest5, earliest_anomaly_timestamp_ASIA_Shanghai

try:
    earliest5
    earliest_anomaly_timestamp_ASIA_Shanghai
except NameError:
    raise RuntimeError("Required variables 'earliest5' and 'earliest_anomaly_timestamp_ASIA_Shanghai' not found in kernel.")

lines = []
lines.append("Earliest anomaly points across selected components (Tomcat/apache/IG/MG) within 2021-03-23 18:00–18:30 Asia/Shanghai:")
lines.append("")
# Show up to 5 rows from earliest5
for i, r in enumerate(earliest5.itertuples(index=False), start=1):
    ts = getattr(r, 'timestamp_ASIA/Shanghai') if 'timestamp_ASIA/Shanghai' in earliest5.columns else r[0]
    # Access by position if column name contains '/'
    if 'timestamp_ASIA/Shanghai' in earliest5.columns:
        ts = r._asdict().get('timestamp_ASIA/Shanghai')
        cmdb = r._asdict().get('cmdb_id')
        kpi = r._asdict().get('kpi_name')
        val = r._asdict().get('value')
        gp95 = r._asdict().get('global_P95')
        pct = r._asdict().get('percent_over')
    else:
        ts, cmdb, kpi, val, gp95, pct = r
    kpi_short = (kpi[:120] + '...') if isinstance(kpi, str) and len(kpi) > 120 else str(kpi)
    val_s = f"{val:.6g}" if pd.notna(val) else "NaN"
    gp95_s = f"{gp95:.6g}" if pd.notna(gp95) else "NaN"
    pct_s = f"{pct:.3f}" if pd.notna(pct) else "NaN"
    lines.append(f"{i}. {ts} | {cmdb} | {kpi_short} | value={val_s} | global_P95={gp95_s} | percent_over={pct_s}")

lines.append("")
lines.append(f"Very earliest anomaly timestamp among these components: {earliest_anomaly_timestamp_ASIA_Shanghai}")
lines.append("")
lines.append("Note: percent_over = (value - global_P95) / global_P95 (NaN if global_P95 is zero or missing).")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name    global_P95
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1         IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  4.169736e+01
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.559407e+08
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.865392e+07
4         IG01          JVM-Operating System_7778_JVM_JVM_CPULoad  3.978100e-01
...        ...                                                ...           ...
1152  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  0.000000e+00
1153  apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...  1.300000e+01
1154  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps  0.000000e+00
1155  apache02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.200000e+01
1156  apache02       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.776720e+01

[1157 rows x 3 columns],         timestamp   cmdb_id                                      kpi_name         value
0      1616493600      IG01      JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1      1616493600      IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.147830e+01
2      1616493600      IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.232846e+08
3      1616493600      IG01   JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.865392e+07
4      1616493600      IG01      JVM-Threads_7778_JVM_ThreadCount_Threads  7.600000e+01
...           ...       ...                                           ...           ...
21833  1616495400  apache01                    OSLinux-CPU_CPU_CPUCpuUtil  2.581400e+01
21834  1616495400  apache01  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  7.300000e+01
21835  1616495400  apache01  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  3.053440e+01
21836  1616495400  apache02                    OSLinux-CPU_CPU_CPUCpuUtil  5.943000e-01
21837  1616495400  apache02  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.773540e+01

[21838 rows x 4 columns],         timestamp   cmdb_id                                      kpi_name         value    global_P95  is_anomaly
0      1616493600      IG01      JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09        True
1      1616493600      IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  3.147830e+01  4.169736e+01       False
2      1616493600      IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.232846e+08  8.559407e+08       False
3      1616493600      IG01   JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.865392e+07  5.865392e+07        True
4      1616493600      IG01      JVM-Threads_7778_JVM_ThreadCount_Threads  7.600000e+01  7.550000e+01        True
...           ...       ...                                           ...           ...           ...         ...
21833  1616495400  apache01                    OSLinux-CPU_CPU_CPUCpuUtil  2.581400e+01  2.577971e+01        True
21834  1616495400  apache01  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  7.300000e+01  7.300000e+01        True
21835  1616495400  apache01  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  3.053440e+01  3.053440e+01        True
21836  1616495400  apache02                    OSLinux-CPU_CPU_CPUCpuUtil  5.943000e-01  7.929600e-01       False
21837  1616495400  apache02  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.773540e+01  2.776720e+01       False

[21838 rows x 6 columns],         timestamp   cmdb_id                                           kpi_name         value    global_P95  is_anomaly
0      1616493600      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09        True
3      1616493600      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.865392e+07  5.865392e+07        True
4      1616493600      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads  7.600000e+01  7.550000e+01        True
11     1616493600      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps  0.000000e+00  0.000000e+00        True
12     1616493600      IG01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01  9.800000e+01        True
...           ...       ...                                                ...           ...           ...         ...
21828  1616495400  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  5.000000e+02        True
21830  1616495400  apache01                    OSLinux-CPU_CPU-1_SingleCpuUtil  4.045170e+01  4.045170e+01        True
21833  1616495400  apache01                         OSLinux-CPU_CPU_CPUCpuUtil  2.581400e+01  2.577971e+01        True
21834  1616495400  apache01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  7.300000e+01  7.300000e+01        True
21835  1616495400  apache01       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  3.053440e+01  3.053440e+01        True

[11889 rows x 6 columns],         timestamp   cmdb_id                                           kpi_name         value    global_P95  is_anomaly  percent_over  timestamp_ASIA/Shanghai
0      1616493600      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09        True      0.000000  2021-03-23 18:00:00 CST
3      1616493600      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.865392e+07  5.865392e+07        True      0.000000  2021-03-23 18:00:00 CST
4      1616493600      IG01           JVM-Threads_7778_JVM_ThreadCount_Threads  7.600000e+01  7.550000e+01        True      0.006623  2021-03-23 18:00:00 CST
11     1616493600      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps  0.000000e+00  0.000000e+00        True           NaN  2021-03-23 18:00:00 CST
12     1616493600      IG01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01  9.800000e+01        True      0.000000  2021-03-23 18:00:00 CST
...           ...       ...                                                ...           ...           ...         ...           ...                      ...
21828  1616495400  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  5.000000e+02        True      0.000000  2021-03-23 18:30:00 CST
21830  1616495400  apache01                    OSLinux-CPU_CPU-1_SingleCpuUtil  4.045170e+01  4.045170e+01        True      0.000000  2021-03-23 18:30:00 CST
21833  1616495400  apache01                         OSLinux-CPU_CPU_CPUCpuUtil  2.581400e+01  2.577971e+01        True      0.001330  2021-03-23 18:30:00 CST
21834  1616495400  apache01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  7.300000e+01  7.300000e+01        True      0.000000  2021-03-23 18:30:00 CST
21835  1616495400  apache01       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  3.053440e+01  3.053440e+01        True      0.000000  2021-03-23 18:30:00 CST

[6320 rows x 8 columns],         timestamp   cmdb_id                                           kpi_name        value   global_P95  is_anomaly  percent_over  timestamp_ASIA/Shanghai
54     1616493600      MG01        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity    15.417000    15.417000        True           0.0  2021-03-23 18:00:00 CST
55     1616493600      MG01  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...     0.471500     0.471500        True           0.0  2021-03-23 18:00:00 CST
56     1616493600      MG01       OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace   140.000000   140.000000        True           0.0  2021-03-23 18:00:00 CST
57     1616493600      MG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvail...  3730.000000  3730.000000        True           0.0  2021-03-23 18:00:00 CST
58     1616493600      MG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity     0.426800     0.426800        True           0.0  2021-03-23 18:00:00 CST
...           ...       ...                                                ...          ...          ...         ...           ...                      ...
21825  1616495400  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     0.000000     0.000000        True           NaN  2021-03-23 18:30:00 CST
21824  1616495400  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     0.000000     0.000000        True           NaN  2021-03-23 18:30:00 CST
21761  1616495400  Tomcat04  OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSIn...     0.004196     0.004196        True           0.0  2021-03-23 18:30:00 CST
21734  1616495400  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter     0.000000     0.000000        True           NaN  2021-03-23 18:30:00 CST
21683  1616495400  Tomcat03  OSLinux-OSLinux_FILESYSTEM_----tmp_FSInodeUsed...     0.014343     0.014343        True           0.0  2021-03-23 18:30:00 CST

[6320 rows x 8 columns],     timestamp_ASIA/Shanghai cmdb_id                                           kpi_name      value  global_P95  percent_over
54  2021-03-23 18:00:00 CST    MG01        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity    15.4170     15.4170           0.0
55  2021-03-23 18:00:00 CST    MG01  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...     0.4715      0.4715           0.0
56  2021-03-23 18:00:00 CST    MG01       OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace   140.0000    140.0000           0.0
57  2021-03-23 18:00:00 CST    MG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvail...  3730.0000   3730.0000           0.0
58  2021-03-23 18:00:00 CST    MG01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity     0.4268      0.4268           0.0, '2021-03-23 18:00:00 CST')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz, datetime

# Reuse existing variables if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# 1) Identify Tomcat01 KPI names containing 'Tomcat-Threads' or 'Tomcat-Requests'
kpis_tomcat01_all = df_metric.loc[df_metric['cmdb_id'] == 'Tomcat01', 'kpi_name'].dropna().unique()
mask_threads_requests = [("Tomcat-Threads" in k) or ("Tomcat-Requests" in k) for k in kpis_tomcat01_all]
kpis_tomcat01 = list(pd.Series(kpis_tomcat01_all)[mask_threads_requests])

# 2) Compute global_P95 for these KPIs using full metric.csv (reuse 'thresholds' if available)
if 'thresholds' in globals():
    thresholds_full = thresholds.copy()
else:
    thresholds_full = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value': 'global_P95'})
    )

# Filter thresholds to Tomcat01 selected KPIs
thr_tomcat01 = thresholds_full[(thresholds_full['cmdb_id'] == 'Tomcat01') & (thresholds_full['kpi_name'].isin(kpis_tomcat01))].copy()

# 3) Filter Tomcat01 KPI points to incident window (inclusive)
start_local = tz.localize(datetime.datetime(2021,3,23,18,0,0))
end_local = tz.localize(datetime.datetime(2021,3,23,18,30,0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts = int(end_local.astimezone(pytz.UTC).timestamp())

df_tomcat01_window = df_metric[
    (df_metric['cmdb_id'] == 'Tomcat01') &
    (df_metric['kpi_name'].isin(kpis_tomcat01)) &
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts)
].copy()

# Merge global_P95 into windowed data
df_tomcat01_w = df_tomcat01_window.merge(thr_tomcat01, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies where value >= global_P95 (only when global_P95 is present and not zero? keep comparison; treat gp95==0 -> still compare)
mask_has_gp = df_tomcat01_w['global_P95'].notna()
df_tomcat01_w['is_anomaly'] = False
df_tomcat01_w.loc[mask_has_gp, 'is_anomaly'] = df_tomcat01_w.loc[mask_has_gp, 'value'] >= df_tomcat01_w.loc[mask_has_gp, 'global_P95']

# 3) Aggregate per KPI for required fields
# earliest_timestamp_in_window_where value >= global_P95 (epoch int), max_value_within_window, timestamp_of_max_value_within_window (epoch int), anomaly_count_within_window
agg_funcs = []
grouped = df_tomcat01_w.groupby(['cmdb_id','kpi_name'], as_index=False)

agg_df = grouped.agg(
    global_P95 = ('global_P95', 'first'),
    anomaly_count_within_window = ('is_anomaly', 'sum'),
    max_value_within_window = ('value', 'max'),
)

# Compute timestamp_of_max_value_within_window
max_ts = (
    df_tomcat01_w.loc[df_tomcat01_w.groupby(['cmdb_id','kpi_name'])['value'].idxmax().dropna()]
    .loc[:, ['cmdb_id','kpi_name','timestamp']]
    .rename(columns={'timestamp':'timestamp_of_max_value_within_window'})
)
agg_df = agg_df.merge(max_ts, on=['cmdb_id','kpi_name'], how='left')

# Compute earliest anomaly timestamp per group
anom_rows = df_tomcat01_w[df_tomcat01_w['is_anomaly']]
if not anom_rows.empty:
    earliest_anom = (
        anom_rows.groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anomaly_ts'})
    )
else:
    earliest_anom = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_ts'])

agg_df = agg_df.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Format timestamps to Asia/Shanghai strings (or None)
def fmt_ts_col(ts_series):
    return ts_series.dropna().astype(int).apply(lambda x: pd.to_datetime(x, unit='s', utc=True).tz_convert(tz).strftime("%Y-%m-%d %H:%M:%S %Z"))

agg_df['earliest_timestamp_in_window_where_value>=global_P95'] = None
mask_e = agg_df['earliest_anomaly_ts'].notna()
if mask_e.any():
    agg_df.loc[mask_e,'earliest_timestamp_in_window_where_value>=global_P95'] = fmt_ts_col(agg_df.loc[mask_e,'earliest_anomaly_ts'])

agg_df['timestamp_of_max_value_within_window'] = None
mask_m = agg_df['timestamp_of_max_value_within_window'].notna()
if mask_m.any():
    agg_df.loc[mask_m,'timestamp_of_max_value_within_window'] = fmt_ts_col(agg_df.loc[mask_m,'timestamp_of_max_value_within_window'])

# Select and order required columns, limit to 20 rows
tomcat01_kpi_summary = agg_df[[
    'cmdb_id',
    'kpi_name',
    'global_P95',
    'earliest_timestamp_in_window_where_value>=global_P95',
    'max_value_within_window',
    'timestamp_of_max_value_within_window',
    'anomaly_count_within_window'
]].copy().head(20)

# 4) Compute single earliest timestamp among these KP95 crossings for Tomcat01 (minimum of earliest_anomaly_ts)
valid_earliest_ts = agg_df['earliest_anomaly_ts'].dropna().astype(int)
if not valid_earliest_ts.empty:
    earliest_tomcat01_crossing = pd.to_datetime(int(valid_earliest_ts.min()), unit='s', utc=True).tz_convert(tz).strftime("%Y-%m-%d %H:%M:%S %Z")
else:
    earliest_tomcat01_crossing = None

# Keep intermediate tables stored: thr_tomcat01, df_tomcat01_window, df_tomcat01_w, agg_df
thr_tomcat01, df_tomcat01_window, df_tomcat01_w, agg_df, tomcat01_kpi_summary, earliest_tomcat01_crossing

```
Out[7]:
```
```python
# Summarize Tomcat01 thread/request KPIs using previously computed tomcat01_kpi_summary and earliest_tomcat01_crossing.
# Reuse kernel variables and produce a compact plain-English summary string.

try:
    tomcat01_kpi_summary
    earliest_tomcat01_crossing
except NameError:
    raise RuntimeError("Required variables 'tomcat01_kpi_summary' and 'earliest_tomcat01_crossing' not found in kernel.")

lines = []
lines.append("Tomcat01 — Thread/Request KPI summary (window 2021-03-23 18:00–18:30 Asia/Shanghai):")
lines.append("")
# Header
lines.append("Columns: cmdb_id | kpi_name | global_P95 | earliest_crossing | max_in_window | ts_of_max | anomaly_count")
for i, r in tomcat01_kpi_summary.reset_index(drop=True).iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    # truncate kpi name if very long
    kpi_disp = (kpi if len(kpi) <= 120 else kpi[:120] + '...')
    gp95 = f"{r['global_P95']:.6g}" if pd.notna(r['global_P95']) else "NaN"
    earliest = r['earliest_timestamp_in_window_where_value>=global_P95'] if r['earliest_timestamp_in_window_where_value>=global_P95'] is not None else "None"
    maxv = f"{r['max_value_within_window']:.6g}" if pd.notna(r['max_value_within_window']) else "No data"
    ts_of_max = r['timestamp_of_max_value_within_window'] if r['timestamp_of_max_value_within_window'] is not None else "None"
    ac = int(r['anomaly_count_within_window']) if pd.notna(r['anomaly_count_within_window']) else 0
    lines.append(f"- {cmdb} | {kpi_disp} | global_P95={gp95} | earliest_crossing={earliest} | max={maxv} | ts_of_max={ts_of_max} | anomalies={ac}")

lines.append("")
lines.append(f"Earliest Tomcat01 KP95 crossing in window: {earliest_tomcat01_crossing}")
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name    global_P95
754  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.600000e+02
755  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.978080e+05
756  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.729649e+09
757  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.822570e+06
767  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02
768  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  3.000000e+00
769  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02,         timestamp   cmdb_id                                           kpi_name         value
540    1616493600  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.600000e+02
541    1616493600  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.978080e+05
542    1616493600  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.727436e+09
543    1616493600  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.815012e+06
553    1616493600  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02
...           ...       ...                                                ...           ...
21564  1616495400  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.729715e+09
21565  1616495400  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.822898e+06
21575  1616495400  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02
21576  1616495400  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+00
21577  1616495400  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02

[217 rows x 4 columns],       timestamp   cmdb_id                                           kpi_name         value    global_P95  is_anomaly
0    1616493600  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.600000e+02  5.600000e+02        True
1    1616493600  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.978080e+05  3.978080e+05        True
2    1616493600  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.727436e+09  1.729649e+09       False
3    1616493600  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.815012e+06  2.822570e+06       False
4    1616493600  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02  1.000000e+02        True
..          ...       ...                                                ...           ...           ...         ...
212  1616495400  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.729715e+09  1.729649e+09        True
213  1616495400  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.822898e+06  2.822570e+06        True
214  1616495400  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02  1.000000e+02        True
215  1616495400  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+00  3.000000e+00       False
216  1616495400  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  5.000000e+02        True

[217 rows x 6 columns],     cmdb_id                                           kpi_name    global_P95  anomaly_count_within_window  max_value_within_window timestamp_of_max_value_within_window  earliest_anomaly_ts earliest_timestamp_in_window_where_value>=global_P95
0  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.600000e+02                           31             5.600000e+02                                 None           1616493600                            2021-03-23 18:00:00 CST  
1  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.978080e+05                           31             3.978080e+05                                 None           1616493600                            2021-03-23 18:00:00 CST  
2  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.729649e+09                            2             1.729715e+09                                 None           1616495340                            2021-03-23 18:29:00 CST  
3  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.822570e+06                            2             2.822898e+06                                 None           1616495340                            2021-03-23 18:29:00 CST  
4  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02                           31             1.000000e+02                                 None           1616493600                            2021-03-23 18:00:00 CST  
5  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  3.000000e+00                            9             5.000000e+00                                 None           1616493600                            2021-03-23 18:00:00 CST  
6  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                           31             5.000000e+02                                 None           1616493600                            2021-03-23 18:00:00 CST  ,     cmdb_id                                           kpi_name    global_P95 earliest_timestamp_in_window_where_value>=global_P95  max_value_within_window timestamp_of_max_value_within_window  anomaly_count_within_window
0  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.600000e+02                            2021-03-23 18:00:00 CST               5.600000e+02                                 None                           31
1  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.978080e+05                            2021-03-23 18:00:00 CST               3.978080e+05                                 None                           31
2  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.729649e+09                            2021-03-23 18:29:00 CST               1.729715e+09                                 None                            2
3  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.822570e+06                            2021-03-23 18:29:00 CST               2.822898e+06                                 None                            2
4  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02                            2021-03-23 18:00:00 CST               1.000000e+02                                 None                           31
5  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  3.000000e+00                            2021-03-23 18:00:00 CST               5.000000e+00                                 None                            9
6  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                            2021-03-23 18:00:00 CST               5.000000e+02                                 None                           31, '2021-03-23 18:00:00 CST')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```